In [1]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier, RandomForestRegressor, VotingClassifier, VotingRegressor)
from sklearn.linear_model import ElasticNet, LinearRegression, LogisticRegression
from sklearn.metrics import (RocCurveDisplay, accuracy_score, classification_report, confusion_matrix, f1_score, log_loss, precision_score, r2_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import (LabelEncoder, MinMaxScaler, OneHotEncoder, PolynomialFeatures, StandardScaler)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from tqdm import tqdm

In [2]:
data_path = r"D:\Machine_Learning_CDAC-AI\a] Datasets\BreastCancer.csv"
cancer = pd.read_csv(data_path)

In [3]:
X = cancer.drop("Class", axis=1)
y = cancer["Class"]

final_y = y
if y.dtype == "object":
    le = LabelEncoder()
    final_y = le.fit_transform(y)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, final_y, test_size=0.3, random_state=26, stratify=final_y
)

In [5]:
features = [2, 3, 4, 5, 6, 7]
estimators = [25, 50, 75, 100, 125]
scores = []

for f in tqdm(features, desc="Tuning max_features"):
    for n in estimators:
        rf = RandomForestClassifier(
            random_state=26, max_features=f, n_estimators=n
        )
        rf.fit(X_train, y_train)
        y_pred_prob = rf.predict_proba(X_test)
        loss = log_loss(y_test, y_pred_prob)
        scores.append([f, n, loss])

Tuning max_features: 100%|███████████████████████████████████████████████████████████████| 6/6 [00:05<00:00,  1.16it/s]


In [6]:
df_score = pd.DataFrame(scores, columns=["max_features", "n_est", "log_loss"])
df_score = df_score.sort_values("log_loss").reset_index(drop=True)

print(df_score.head())

best_f = int(df_score.loc[0, "max_features"])
best_n = int(df_score.loc[0, "n_est"])

print(
    f"\nBest Model Selected: max_features={best_f}, n_estimators={best_n} with Log Loss={df_score.loc[0, 'log_loss']:.4f}"
)

   max_features  n_est  log_loss
0             2     25  0.094429
1             2    125  0.096572
2             6     25  0.097490
3             2    100  0.099016
4             2     75  0.100390

Best Model Selected: max_features=2, n_estimators=25 with Log Loss=0.0944


In [7]:
bm = RandomForestClassifier(
    random_state=26, max_features=best_f, n_estimators=best_n
)
bm.fit(X_train, y_train)
model_path = r"D:\Machine_Learning_CDAC-AI\P] Deployment & Production\A] Model Persistence\BreastCancer_model.pkl"

with open(model_path, "wb") as pkfile:
    pickle.dump(bm, pkfile)

print(f"\nModel successfully saved to: {model_path}")


Model successfully saved to: D:\Machine_Learning_CDAC-AI\P] Deployment & Production\A] Model Persistence\BreastCancer_model.pkl
